### Bayesian MC dropout query strategies

Bayesian query strategies use Monte Carlo (MC) dropout to approximate uncertainty in deep learning models. This works by computing multiple forward passes through a neural network with the dropout layers activated. For this example we are going to use a subset of [_MNIST_](https://archive.ics.uci.edu/dataset/683/mnist+database+of+handwritten+digits) dataset, loaded from _sklearn_.

In [6]:
import numpy as np
import torch
from skorch import NeuralNetClassifier
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from activelearning.AL_cycle import plot_results, strategy_comparison
from activelearning.queries.bayesian.mc_max_entropy import mc_max_entropy
from activelearning.queries.representative.kmeans_query import query_kmeans_foreach
from activelearning.queries.representative.coreset_query import query_coreset
from activelearning.queries.representative.random_query import query_random
from activelearning.utils.skorch_nnet import reshapedVGG

torch.manual_seed(123)
np.random.seed(123)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import sys
import os
import numpy as np
import torch

from embeddings.embedding import MaskedReconstruction, MultimodalMAE

PROJECT_ROOT = os.path.abspath("../..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from al_data_process.data_preprocess import create_windowed_ts, map_class_label_to_idx
from al_data_process.dataset import stratified_sampling, stratified_sampling_train_val

window_length = 0.4
overlap = 0.25

invalid_classes = [14]
Xt_acc, Xt_gyr, Xt_mag, Xt_mic, y_org_label = create_windowed_ts(data_path="../../tool-tracking-data/",
                                                                     tool="electric_screwdriver",
                                                                     invalid_classes=invalid_classes,
                                                                     window_length = window_length,
                                                                     overlap = overlap)

all_classes, counts = np.unique(y_org_label, return_counts=True)

Xt_acc = Xt_acc[:, :, 1:].astype(np.float32)
Xt_gyr = Xt_gyr[:, :, 1:].astype(np.float32)
Xt_mag = Xt_mag[:, :, 1:].astype(np.float32)
Xt_mic = Xt_mic[:, :, 1:].astype(np.float32)

y = map_class_label_to_idx(y_org_label)

train_set, test_set = stratified_sampling_train_val(torch.from_numpy(Xt_acc),
                                                  torch.from_numpy(Xt_gyr),
                                                  torch.from_numpy(Xt_mag),
                                                  torch.from_numpy(Xt_mic),
                                                  torch.tensor(y))


X_pool = np.empty(len(train_set), dtype=object)
y_pool = np.empty(len(train_set))
for i in range(len(train_set)):
    X_pool[i] = train_set[i][0]
    y_pool[i] = train_set[i][1]


X_test = np.empty(len(test_set), dtype=object)
y_test = np.empty(len(test_set))
for i in range(len(test_set)):
    X_test[i] = test_set[i][0]
    y_test[i] = test_set[i][1]


model = MultimodalMAE().to(device)
recon = MaskedReconstruction().to(device)

checkpoint = torch.load("../embeddings/multimodal_mae.pth", map_location=device)

model.load_state_dict(checkpoint['model_state_dict'])
recon.load_state_dict(checkpoint['recon_state_dict'])

model.eval()
recon.eval()

print("Models loaded and ready for inference.")

@torch.no_grad()
def extract_embedding(acc, gyr, mag, mic):
    model.eval()
    acc, gyr, mag, mic = (
        acc.to(device),
        gyr.to(device),
        mag.to(device),
        mic.to(device).squeeze()
    )
    z, _ = model(acc, gyr, mag, mic)
    return z.cpu()


embeddings = []
for acc, gyr, mag, mic in X_pool:
    z = extract_embedding(acc, gyr, mag, mic)
    embeddings.append(z)

X_pool_embeddings = np.array(embeddings)

[INFO] Preparing data from:
  ../../tool-tracking-data/electric_screwdriver/pythagoras-10-20200716
  ../../tool-tracking-data/electric_screwdriver/pythagoras-10-20200716
  ../../tool-tracking-data/electric_screwdriver/pythagoras-10-20200716
  ../../tool-tracking-data/electric_screwdriver/pythagoras-10-20200716


[INFO] Read data:   0%|          | 0/16 [00:00<?, ?it/s]

[WARN] The mean sampling rate from the filename (../../tool-tracking-data/electric_screwdriver/pythagoras-10-20200716/ACC-03-102.291.csv) and from the timestamps differ by 0.001Hz



[INFO] Read data:  12%|█▎        | 2/16 [00:00<00:01,  7.42it/s, file=MIC-03-8000.csv]

[WARN] The mean sampling rate from the filename (../../tool-tracking-data/electric_screwdriver/pythagoras-10-20200716/MAG-01-155.087.csv) and from the timestamps differ by 0.001Hz

[WARN] The mean sampling rate from the filename (../../tool-tracking-data/electric_screwdriver/pythagoras-10-20200716/GYR-03-102.291.csv) and from the timestamps differ by 0.001Hz



...

[WARN] The mean sampling rate from the filename (../../tool-tracking-data/electric_screwdriver/pythagoras-10-20200716/GYR-02-102.291.csv) and from the timestamps differ by 0.001Hz



[INFO] Read annotation: 100%|██████████| 16/16 [00:01<00:00, 17.25it/s, file=data-02.annotation]

[WARN] The mean sampling rate from the filename (../../tool-tracking-data/electric_screwdriver/pythagoras-10-20200716/ACC-01-102.291.csv) and from the timestamps differ by 0.001Hz

[WARN] The mean sampling rate from the filename (../../tool-tracking-data/electric_screwdriver/pythagoras-10-20200716/MAG-02-154.679.csv) and from the timestamps differ by 0.001Hz

[WARN] The mean sampling rate from the filename (../../tool-tracking-data/electric_screwdriver/pythagoras-10-20200716/GYR-01-102.291.csv) and from the timestamps differ by 0.001Hz

[WARN] The mean sampling rate from the filename (../../tool-tracking-data/electric_screwdriver/pythagoras-10-20200716/ACC-04-102.291.csv) and from the timestamps differ by 0.001Hz

[WARN] The mean sampling rate from the filename (../../tool-tracking-data/electric_screwdriver/pythagoras-10-20200716/ACC-02-102.291.csv) and from the timestamps differ by 0.001Hz

[WARN] The mean sampling rate from the filename (../../tool-tracking-data/electric_screwdriver/

[INFO] Read annotation: 100%|██████████| 16/16 [00:01<00:00, 11.33it/s, file=data-02.annotation]


[INFO] Finished with 4 measurement(s).
[INFO] segment ['acc' 'gyr' 'mag' 'mic'] together
[INFO] segment ['acc' 'gyr' 'mag' 'mic'] together
[INFO] segment ['acc' 'gyr' 'mag' 'mic'] together
[INFO] segment ['acc' 'gyr' 'mag' 'mic'] together
Segment: 100%|██████████| 19380/19380 [00:00<00:00, 108128.80it/s]
[2 3 4 5 6 7 8] [ 546  234  199   89   50   59 3606]
count_1_14: 3, count_invalid_classes: 8, count_maj_vote: 46, count_diff_label: 5
Models loaded and ready for inference.


We evaluate the performance of the VGG classifier that we are going to use on the complete training set. This will serve as reference metric for the active learning query strategies, as we want to reach the same accuracy but with less labeled data.

In [12]:
num_lstm_layers = 1
hidden_size = 32
dropout_p = 0.3
num_classes = 7
lr=2e-3
batch_size=16

In [ ]:
from model.lstm_classif import LSTM_CLASSIF
from sklearn.metrics import f1_score
# from skorch.helper import predefined_split
from skorch.dataset import CVSplit
# from skorch.callbacks import Checkpoint
from skorch.callbacks import EpochScoring, Checkpoint


def compute_class_weights_from_labels(y, beta=0.999):
    """
    Compute effective number class weights directly from labels.

    Args:
        y: array-like, shape (num_samples,)
           Class labels (int or float, discrete)
        beta: float, smoothing factor

    Returns:
        weights: np.array, shape (num_classes,)
    """
    classes, counts = np.unique(y, return_counts=True)
    classes = classes.astype(int)

    counts = counts.astype(np.float32)
    counts = np.maximum(counts, 1.0)

    effective_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / effective_num

    weights = weights / weights.mean()

    num_classes = int(np.max(classes)) + 1
    full_weights = np.zeros(num_classes, dtype=np.float32)
    full_weights[classes] = weights

    return full_weights


class_weights = compute_class_weights_from_labels(y_pool, beta=0.999)
class_weights_tensor = torch.from_numpy(class_weights).float()


def collate_fn(batch):
    x_batch, y_batch = zip(*batch)  # x_batch = list of tuples, y_batch = list of labels

    acc, gyr, mag, mic = zip(*x_batch)  # each is a tuple of tensors

    acc = torch.stack(acc, dim=0)  # shape: (batch, time, features)
    gyr = torch.stack(gyr, dim=0)
    mag = torch.stack(mag, dim=0)
    mic = torch.stack(mic, dim=0)

    if isinstance(y_batch[0], torch.Tensor):
        y_batch = torch.stack(y_batch).long()  # works if labels are 0-dim tensors
    else:
        y_batch = torch.tensor(y_batch, dtype=torch.long)


    return (acc, gyr, mag, mic), y_batch


# classifier = NeuralNetClassifier(
#     LSTM_CLASSIF(input_size=3, hidden_size=hidden_size, num_layers=num_lstm_layers, num_classes=num_classes,
#                              dropout_p=dropout_p),
#     criterion=torch.nn.CrossEntropyLoss,
#     criterion__weight=class_weights_tensor,
#     lr=2e-3,
#     batch_size=16,
#     iterator_train__collate_fn=collate_fn,
#     iterator_valid__collate_fn=collate_fn,
#     optimizer=torch.optim.Adam,
#     train_split=None,
#     max_epochs=100,
#     device=device,
#     # train_loss_best=True,
# )


f1_cb = EpochScoring(
    scoring='f1_macro',      # sklearn scorer name
    lower_is_better=False,
    name='valid_f1',         # metric name in history
    on_train=False,          # compute on validation set
)

checkpoint = Checkpoint(
    monitor='valid_f1_best',
    # load_best=True,
    f_params='best_weights.pt',
)


classifier = NeuralNetClassifier(
    LSTM_CLASSIF(
        input_size=3,
        hidden_size=hidden_size,
        num_layers=num_lstm_layers,
        num_classes=num_classes,
        dropout_p=dropout_p,
    ),
    criterion=torch.nn.CrossEntropyLoss,
    criterion__weight=class_weights_tensor,
    optimizer=torch.optim.Adam,
    lr=lr,
    batch_size=batch_size,
    max_epochs=100,
    device=device,

    # train_split=ValidSplit(0.2, stratified=True),
    # train_split=CVSplit(0.2, stratified=True),
    train_split=CVSplit(0.15 / 0.85, stratified=True, random_state=42),

    iterator_train__collate_fn=collate_fn,
    iterator_valid__collate_fn=collate_fn,

    callbacks=[f1_cb, checkpoint],
)

classifier.fit(X_pool, y_pool)

classifier.load_params(f_params='best_weights.pt')

goal_acc = classifier.score(X_test, y_test)

y_pred = classifier.predict(X_test)
# print(y_pred.shape)
goal_f1 = f1_score(y_test, y_pred, average="macro")

print(f"Goal Accuracy: {goal_acc}")
print(f"Goal F1 Score: {goal_f1}")

  epoch    train_loss    valid_acc    valid_f1    valid_loss    cp     dur
-------  ------------  -----------  ----------  ------------  ----  ------
      1        1.5069       0.7646      0.3826        0.9995     +  1.4114
      2        1.0451       0.7075      0.4334        0.8427     +  1.4171
      3        0.9037       0.7688      0.4677        0.7539     +  1.4067
      4        0.8306       0.7744      0.5174        0.7546     +  1.3571
      5        0.8012       0.7786      0.4995        0.7700        1.4255
      6        0.7648       0.7799      0.5209        0.6998     +  1.3667
      7        0.7524       0.7953      0.5568        0.7005     +  1.3988
      8        0.7018       0.7591      0.5124        0.8017        1.3893
      9        0.7569       0.7869      0.5772        0.6968     +  1.3683
     10        0.6994       0.7911      0.5699        0.6638        1.3782
     11        0.6769       0.8022      0.5868        0.6618     +  1.3757
     12        0.6641    

In [14]:
n_init_samples = 400 # number of initial training samples
selected_idx = np.random.choice(len(X_pool), size=n_init_samples, replace=False)

X_initial = X_pool[selected_idx]
y_initial = y_pool[selected_idx]
X_initial_emb = X_pool_embeddings[selected_idx]

X_pool = np.delete(X_pool, selected_idx, axis=0)
y_pool = np.delete(y_pool, selected_idx, axis=0)
X_pool_embeddings = np.delete(X_pool_embeddings, selected_idx, axis=0)

To compare how different query strategies perform, we can use the *strategy_comparison* function and pass the strategies to be used. We can also pass more than one number of instances, to check whether a different batch size influences performance. *plot_results* can be used to immediatly plot the output from *strategy_comparison*, or a custom graph can be created from the scores data frame.

In [ ]:


n_instances = [100]
goal_metric = "f1"
# goal_metric = "acc"

# classifier = NeuralNetClassifier(
#     LSTM_CLASSIF(input_size=3, hidden_size=hidden_size, num_layers=num_lstm_layers, num_classes=num_classes,
#                              dropout_p=dropout_p),
#     criterion=torch.nn.CrossEntropyLoss,
#     # criterion__weight=class_weights_tensor,
#     lr=2e-3,
#     batch_size=16,
#     iterator_train__collate_fn=collate_fn,
#     iterator_valid__collate_fn=collate_fn,
#     optimizer=torch.optim.Adam,
#     train_split=None,
#     max_epochs=100,
#     device=device,
# )


f1_cb_al = EpochScoring(
    scoring='f1_macro',      # sklearn scorer name
    lower_is_better=False,
    name='valid_f1',         # metric name in history
    on_train=False,          # compute on validation set
)

checkpoint_al = Checkpoint(
    monitor='valid_f1_best',
    # load_best=True,
    f_params='best_weights.pt',
)

classifier = NeuralNetClassifier(
    LSTM_CLASSIF(
        input_size=3,
        hidden_size=hidden_size,
        num_layers=num_lstm_layers,
        num_classes=num_classes,
        dropout_p=dropout_p,
    ),
    criterion=torch.nn.CrossEntropyLoss,
    # criterion__weight=class_weights_tensor,
    optimizer=torch.optim.Adam,
    lr=2e-3,
    batch_size=16,
    max_epochs=100,
    device=device,

    # Use stratified validation split
    # train_split=ValidSplit(0.2, stratified=True),
    train_split=CVSplit(0.15 / 0.85),

    iterator_train__collate_fn=collate_fn,
    iterator_valid__collate_fn=collate_fn,

    callbacks=[f1_cb_al, checkpoint_al],
)



# classifier.get_default_callbacks()
# classifier.callbacks

scores = strategy_comparison(
    X_train=X_initial_emb,
    y_train=y_initial,
    X_pool=X_pool_embeddings,
    y_pool=y_pool,
    X_test=X_test,
    y_test=y_test,
    X_pool_org=X_pool,
    X_train_org=X_initial,
    # classifier="nnet_bo",
    classifier=classifier,
    # query_strategies=[mc_bald, mc_max_entropy, mc_max_varratios, query_random, mc_max_meanstd],
    query_strategies=[query_coreset, query_kmeans_foreach, query_random],
    # query_strategies=[mc_max_entropy],
    # query_strategies=[mc_max_entropy, mc_bald],
    # n_instances=[32],
    n_instances=n_instances,
    # goal_acc=goal_acc,
    goal_metric=goal_metric, 
    goal_metric_val = goal_f1 if goal_metric == "f1" else goal_acc
    # max_epochs=15,
)

  epoch    train_loss    valid_acc    valid_f1    valid_loss    cp     dur
-------  ------------  -----------  ----------  ------------  ----  ------
      1        1.3151       0.7042      0.1377        1.1966     +  0.1497
      2        0.8374       0.7042      0.1377        0.9722        0.1735
      3        0.7445       0.7042      0.1377        0.8734        0.1432
      4        0.6519       0.7042      0.1377        0.8196        0.1399
      5        0.6117       0.7042      0.1377        0.8208        0.1381
      6        0.5705       0.7887      0.2699        0.7751     +  0.1379
      7        0.5588       0.7887      0.2638        0.7804        0.1777
      8        0.5414       0.7887      0.2699        0.7972        0.1374
      9        0.5081       0.7887      0.2586        0.7917        0.1379
     10        0.4951       0.8310      0.2805        0.7923     +  0.1397
     11        0.5068       0.7887      0.2586        0.8606        0.1896
     12        0.4933    

In [ ]:
print(scores)


plot_results(
    scores,  # output data frame from strategy_comparison
    n_instances=n_instances,
    tot_samples=len(train_set),  # size of the original training set, for scale
    # goal_acc=goal_acc,
    figsize=(21, 8),
    goal_metric=goal_metric,
    goal_metric_val=goal_f1 if goal_metric == "f1" else goal_acc
)
